# Tunda Box: Junior Data Analyst Assessment (solutions notebook)

**Candidate:** Chala Wakshuma

This notebook holds the working for the SQL, Python, data-quality, statistics and business-case questions. Every SQL query is run with `sqlite3` on the sample data and cross-checked against a pandas calculation using `assert`, so the notebook fails loudly if a number is wrong. The written answers (Parts 1, 6 and the plain-language explanations) are in `README.md`, and the reasoning is in `THOUGHT_PROCESS.md`.

**How to run:** Runtime > Restart session and run all (or `Kernel > Restart & Run All`). Only `pandas` and the Python standard library are used.

## 0. Load the sample data

The tables are typed in exactly as shown in the assessment. City values keep their stray case and spaces, and order 1006 stays duplicated, because finding those problems is part of the task.

In [1]:
import math
import random
import sqlite3

import pandas as pd

pd.set_option("display.width", 140)

customers = pd.DataFrame(
    [
        (101, "Nairobi",  "2026-01-14", "Organic"),
        (102, "nairobi",  "2026-02-03", "Referral"),
        (103, "Mombasa",  "2026-06-20", "Paid Social"),
        (104, "Kisumu",   "2026-03-02", "Organic"),
        (105, "Nairobi ", "2026-03-15", "Paid Social"),
        (106, "Nakuru",   "2026-04-01", "Referral"),
        (107, "Mombasa",  "2026-04-18", "Organic"),
        (108, "Nairobi",  "2026-05-09", "Paid Social"),
        (109, "Eldoret",  "2026-06-21", "Organic"),
        (110, "Kisumu",   "2026-07-05", "Paid Social"),
    ],
    columns=["customer_id", "city", "signup_date", "channel"],
)

orders = pd.DataFrame(
    [
        (1001, 101, "2026-06-03", "completed", 2400, "M-Pesa"),
        (1002, 102, "2026-06-05", "completed", 1850, "M-Pesa"),
        (1003, 103, "2026-06-11", "cancelled", 3200, "Card"),
        (1004, 104, "2026-06-18", "completed", 2100, "M-Pesa"),
        (1005, 101, "2026-06-25", "completed", 2650, "Card"),
        (1006, 105, "2026-07-02", "completed", 1500, "M-Pesa"),
        (1006, 105, "2026-07-02", "completed", 1500, "M-Pesa"),   # duplicate row
        (1007, 106, "2026-07-09", "refunded",  2900, "Card"),
        (1008, 102, "2026-07-14", "completed", 3100, "M-Pesa"),
        (1009, 107, "2026-07-21", "completed", None, "M-Pesa"),   # missing amount
        (1010, 101, "2026-07-28", "completed", 2200, "M-Pesa"),
        (1011, 108, "2026-08-04", "completed", 1750, "Card"),
        (1012, 103, "2026-08-09", "completed", 2800, "M-Pesa"),
        (1013, 104, "2026-08-15", "completed", 185000, "M-Pesa"), # outlier
        (1014, 105, "2026-08-19", "cancelled", 1600, "M-Pesa"),
        (1015, 102, "2026-08-27", "completed", 2450, "M-Pesa"),
        (1016, 106, "2026-09-03", "completed", 3300, "Card"),
        (1017, 101, "2026-09-08", "completed", 2750, "M-Pesa"),
        (1018, 108, "2026-09-14", "refunded",  2000, "M-Pesa"),
        (1019, 107, "2026-09-22", "completed", 2350, "Card"),
        (1020, 999, "2026-09-26", "completed", 1900, "M-Pesa"),   # customer 999 does not exist
    ],
    columns=["order_id", "customer_id", "order_date", "status", "amount_kes", "payment_method"],
)
print(len(customers), "customers,", len(orders), "order rows")

10 customers, 21 order rows


## D1. Audit the data first

Before answering anything I check the data for problems, because they change the answers. Each check below prints the evidence.

In [2]:
print("1) Duplicate rows:", int(orders.duplicated().sum()), "-> order_id", orders[orders.duplicated()]["order_id"].tolist())

print("\n2) Raw city labels:", [repr(c) for c in customers["city"]])
print("   After TRIM + title case:", sorted(customers["city"].str.strip().str.title().unique()))

print("\n3) Missing amounts:", orders[orders["amount_kes"].isna()][["order_id", "status"]].to_dict("records"))

completed = orders[(orders["status"] == "completed") & orders["amount_kes"].notna()].drop_duplicates()
typical = completed["amount_kes"].median()
outliers = completed[completed["amount_kes"] > 10 * typical]
print(f"\n4) Typical (median) completed order: KES {typical:,.0f}; orders above 10x that:",
      outliers[["order_id", "amount_kes"]].to_dict("records"))

orphans = orders[~orders["customer_id"].isin(customers["customer_id"])]
print("\n5) Orders whose customer is not in customers:", orphans[["order_id", "customer_id"]].to_dict("records"))

m = orders.merge(customers, on="customer_id")
early = m[m["order_date"] < m["signup_date"]]
print("\n6) Orders dated before the customer signed up:",
      early[["order_id", "order_date", "signup_date"]].to_dict("records"))

print("\n7) Statuses:", orders.drop_duplicates()["status"].value_counts().to_dict())

1) Duplicate rows: 1 -> order_id [1006]

2) Raw city labels: ["'Nairobi'", "'nairobi'", "'Mombasa'", "'Kisumu'", "'Nairobi '", "'Nakuru'", "'Mombasa'", "'Nairobi'", "'Eldoret'", "'Kisumu'"]
   After TRIM + title case: ['Eldoret', 'Kisumu', 'Mombasa', 'Nairobi', 'Nakuru']

3) Missing amounts: [{'order_id': 1009, 'status': 'completed'}]

4) Typical (median) completed order: KES 2,400; orders above 10x that: [{'order_id': 1013, 'amount_kes': 185000.0}]

5) Orders whose customer is not in customers: [{'order_id': 1020, 'customer_id': 999}]

6) Orders dated before the customer signed up: [{'order_id': 1003, 'order_date': '2026-06-11', 'signup_date': '2026-06-20'}]

7) Statuses: {'completed': 16, 'cancelled': 2, 'refunded': 2}


**Decisions from the audit** (all stated in the written answers):
- Remove the duplicate row (order 1006).
- Standardise city names with TRIM and title case.
- Revenue means *completed* orders only; the missing amount is not imputed, it is simply not summed.
- Order 1013 (KES 185,000) is flagged as a probable typo; I show results both as recorded and with it excluded.
- Order 1020 (customer 999) cannot be tied to a city or a customer, so it is left out of city and customer metrics.

## Load into SQLite and create two clean views

`orders_clean` removes the duplicate; `customers_clean` tidies the city labels. All SQL below reads from these views.

In [3]:
con = sqlite3.connect(":memory:")
orders.to_sql("orders", con, index=False)
customers.to_sql("customers", con, index=False)

con.executescript("""
-- orders_clean: removes the duplicated order 1006
CREATE VIEW orders_clean AS
SELECT DISTINCT * FROM orders;

-- customers_clean: trimmed, title-case city (Postgres: INITCAP(TRIM(city)))
CREATE VIEW customers_clean AS
SELECT customer_id,
       UPPER(SUBSTR(TRIM(city),1,1)) || LOWER(SUBSTR(TRIM(city),2)) AS city,
       signup_date, channel
FROM customers;
""")

def sql(query):
    """Run a SQL query on the in-memory database and return a DataFrame."""
    return pd.read_sql_query(query, con)

# the same cleaning in pandas, used to cross-check every SQL result
orders_clean = orders.drop_duplicates().copy()
customers_clean = customers.copy()
customers_clean["city"] = customers_clean["city"].str.strip().str.title()
orders_clean["order_date"] = pd.to_datetime(orders_clean["order_date"])
print(len(orders_clean), "unique orders;", sorted(customers_clean["city"].unique()))

20 unique orders; ['Eldoret', 'Kisumu', 'Mombasa', 'Nairobi', 'Nakuru']


## S1. Orders by status

In [4]:
query_s1 = """
SELECT status, COUNT(*) AS orders
FROM orders_clean
GROUP BY status
ORDER BY orders DESC;
"""
s1 = sql(query_s1)
print(s1.to_string(index=False))

# pandas cross-check
assert dict(zip(s1["status"], s1["orders"])) == orders_clean["status"].value_counts().to_dict()
assert dict(zip(s1["status"], s1["orders"])) == {"completed": 16, "cancelled": 2, "refunded": 2}
print("\nWithout removing the duplicate, 'completed' would wrongly show", int((orders["status"] == "completed").sum()))

   status  orders
completed      16
 refunded       2
cancelled       2

Without removing the duplicate, 'completed' would wrongly show 17


## S2. Revenue by city

(a) Completed revenue by city, as recorded and with the suspicious order 1013 excluded. (b) is a written message, see `README.md`.

In [5]:
query_s2 = """
SELECT c.city,
       SUM(o.amount_kes) AS revenue_kes,
       SUM(CASE WHEN o.amount_kes < 100000 THEN o.amount_kes END) AS revenue_ex_outlier
FROM orders_clean o
JOIN customers_clean c ON c.customer_id = o.customer_id
WHERE o.status = 'completed'
GROUP BY c.city
ORDER BY revenue_ex_outlier DESC;
"""
s2 = sql(query_s2)
print(s2.to_string(index=False))

# pandas cross-check
chk = (orders_clean[orders_clean["status"] == "completed"]
       .merge(customers_clean, on="customer_id")
       .groupby("city")["amount_kes"].sum())
assert chk.to_dict() == dict(zip(s2["city"], s2["revenue_kes"]))
assert dict(zip(s2["city"], s2["revenue_ex_outlier"])) == {"Nairobi": 20650, "Mombasa": 5150, "Nakuru": 3300, "Kisumu": 2100}
print("\nTop city excluding order 1013:", s2.iloc[0]["city"], f"(KES {s2.iloc[0]['revenue_ex_outlier']:,.0f})")

   city  revenue_kes  revenue_ex_outlier
Nairobi      20650.0             20650.0
Mombasa       5150.0              5150.0
 Nakuru       3300.0              3300.0
 Kisumu     187100.0              2100.0

Top city excluding order 1013: Nairobi (KES 20,650)


## S3. Customers who never ordered

In [6]:
query_s3 = """
SELECT c.customer_id, c.city, c.signup_date, c.channel
FROM customers_clean c
LEFT JOIN orders_clean o ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL;
"""
s3 = sql(query_s3)
print(s3.to_string(index=False))

# pandas cross-check
never = customers_clean[~customers_clean["customer_id"].isin(orders_clean["customer_id"])]
assert never["customer_id"].tolist() == s3["customer_id"].tolist() == [109, 110]

 customer_id    city signup_date     channel
         109 Eldoret  2026-06-21     Organic
         110  Kisumu  2026-07-05 Paid Social


## S4. Repeat purchase rate

A repeat buyer has two or more completed orders. The duplicate is removed first (otherwise customer 105 would look like a repeat buyer) and customer 999 is ignored because they are not in the customers table.

In [7]:
query_s4 = """
WITH n AS (
    SELECT customer_id, COUNT(*) AS completed_orders
    FROM orders_clean
    WHERE status = 'completed'
    GROUP BY customer_id
), t AS (
    SELECT (SELECT COUNT(*) FROM customers_clean) AS customers,
           COUNT(*) AS buyers,
           SUM(CASE WHEN n.completed_orders >= 2 THEN 1 ELSE 0 END) AS repeat_buyers
    FROM n
    JOIN customers_clean c ON c.customer_id = n.customer_id   -- drops orphan 999
)
SELECT repeat_buyers, buyers, customers,
       ROUND(100.0 * repeat_buyers / customers, 1) AS pct_of_all_customers,
       ROUND(100.0 * repeat_buyers / buyers, 1)    AS pct_of_buyers
FROM t;
"""
s4 = sql(query_s4)
print(s4.to_string(index=False))

# pandas cross-check
counts = (orders_clean[(orders_clean["status"] == "completed")
                       & orders_clean["customer_id"].isin(customers_clean["customer_id"])]
          .groupby("customer_id").size())
repeat = int((counts >= 2).sum())
assert repeat == 4 and len(counts) == 8 and len(customers_clean) == 10
assert s4.loc[0, "repeat_buyers"] == repeat
print(f"\nRepeat buyers: {sorted(counts[counts >= 2].index.tolist())}")
print(f"Rate among all customers: {repeat / len(customers_clean):.0%}; among buyers: {repeat / len(counts):.0%}")

 repeat_buyers  buyers  customers  pct_of_all_customers  pct_of_buyers
             4       8         10                  40.0           50.0

Repeat buyers: [101, 102, 104, 107]
Rate among all customers: 40%; among buyers: 50%


## S5. Month-on-month revenue

Shown as recorded and with order 1013 excluded. `LAG()` brings the previous month's revenue onto each row.

In [8]:
query_s5 = """
-- Postgres: use DATE_TRUNC('month', order_date) instead of strftime
WITH m AS (
    SELECT strftime('%Y-%m', order_date) AS month,
           SUM(amount_kes) AS revenue_kes,
           SUM(CASE WHEN amount_kes < 100000 THEN amount_kes END) AS revenue_ex_outlier
    FROM orders_clean
    WHERE status = 'completed'
    GROUP BY 1
), p AS (
    SELECT *,
           LAG(revenue_kes) OVER (ORDER BY month)         AS prev,
           LAG(revenue_ex_outlier) OVER (ORDER BY month)  AS prev_ex
    FROM m
)
SELECT month, revenue_kes,
       ROUND(100.0 * (revenue_kes - prev) / prev, 1)                 AS mom_pct,
       revenue_ex_outlier,
       ROUND(100.0 * (revenue_ex_outlier - prev_ex) / prev_ex, 1)    AS mom_pct_ex_outlier
FROM p
ORDER BY month;
"""
s5 = sql(query_s5)
print(s5.to_string(index=False))

# pandas cross-check
done = orders_clean[orders_clean["status"] == "completed"].copy()
done["month"] = done["order_date"].dt.strftime("%Y-%m")
rev = done.groupby("month")["amount_kes"].sum()
rev_ex = done[done["amount_kes"] < 100000].groupby("month")["amount_kes"].sum()
assert rev.tolist() == s5["revenue_kes"].tolist() == [9000, 6800, 192000, 10300]
assert rev_ex.tolist() == s5["revenue_ex_outlier"].tolist() == [9000, 6800, 7000, 10300]
# reconciliation: city totals + the unmatched order 1020 must equal total monthly completed revenue
assert s2["revenue_kes"].sum() + 1900 == s5["revenue_kes"].sum() == 218100
print("\nReconciles: city totals (KES 216,200) + unmatched order 1020 (KES 1,900) = monthly total (KES 218,100)")
print("Excluding the unmatched order 1020, September would be:",
      int(done[(done["month"] == "2026-09") & (done["customer_id"] != 999)]["amount_kes"].sum()))

  month  revenue_kes  mom_pct  revenue_ex_outlier  mom_pct_ex_outlier
2026-06       9000.0      NaN              9000.0                 NaN
2026-07       6800.0    -24.4              6800.0               -24.4
2026-08     192000.0   2723.5              7000.0                 2.9
2026-09      10300.0    -94.6             10300.0                47.1

Reconciles: city totals (KES 216,200) + unmatched order 1020 (KES 1,900) = monthly total (KES 218,100)
Excluding the unmatched order 1020, September would be: 8400


## D2. Spreadsheet formulas: the numbers they should return

The formulas themselves are in `README.md`:
`=SUMIFS(E:E, D:D, "completed", F:F, "M-Pesa")` and `=IFERROR(PROPER(TRIM(XLOOKUP(B2, customers!A:A, customers!B:B))), "Not found")`.
Here I compute what they should return, so the spreadsheet can be checked.

In [9]:
def completed_mpesa(df):
    """Sum of completed M-Pesa orders (what SUMIFS returns; blank amounts are ignored)."""
    mask = (df["status"] == "completed") & (df["payment_method"] == "M-Pesa")
    return df.loc[mask, "amount_kes"].sum()

raw_total = completed_mpesa(orders)
dedup_total = completed_mpesa(orders_clean)
print(f"SUMIFS on the raw sheet            : KES {raw_total:,.0f}")
print(f"After removing the duplicate row   : KES {dedup_total:,.0f}")
print(f"Also excluding order 1013          : KES {dedup_total - 185000:,.0f}")

# the 'city for each order' column, with 'Not found' for orphans
orders_with_city = orders_clean.merge(customers_clean[["customer_id", "city"]], on="customer_id", how="left")
orders_with_city["city"] = orders_with_city["city"].fillna("Not found")
print()
print(orders_with_city[["order_id", "customer_id", "city"]].tail(4).to_string(index=False))
assert (raw_total, dedup_total) == (209550, 208050)

SUMIFS on the raw sheet            : KES 209,550
After removing the duplicate row   : KES 208,050
Also excluding order 1013          : KES 23,050

 order_id  customer_id      city
     1017          101   Nairobi
     1018          108   Nairobi
     1019          107   Mombasa
     1020          999 Not found


## D3. Mean or median

Typical completed order value, with and without the suspicious order.

In [10]:
vals = orders_clean[(orders_clean["status"] == "completed")]["amount_kes"].dropna()
print(f"All completed orders with an amount (n={len(vals)}): mean KES {vals.mean():,.0f}, median KES {vals.median():,.0f}")
ex = vals[vals < 100000]
print(f"Excluding order 1013 (n={len(ex)}):               mean KES {ex.mean():,.0f}, median KES {ex.median():,.0f}")
assert round(vals.mean()) == 14540 and vals.median() == 2400

All completed orders with an amount (n=15): mean KES 14,540, median KES 2,400
Excluding order 1013 (n=14):               mean KES 2,364, median KES 2,375


## T1(a). What a p-value means (demonstration)

A p-value answers: *if there were truly no difference, how often would luck alone produce a gap at least this big?* I can show that directly with the A/B data from T2. Shuffle the 110 conversions randomly between the two groups many times and count how often the gap is as large as the one observed.

In [11]:
random.seed(42)
users_per_group, total_conversions, observed_new = 1000, 110, 60   # 50 + 60 conversions in 2,000 users
observed_gap = abs(observed_new - (total_conversions - observed_new))

runs = 5000
as_extreme = 0
for _ in range(runs):
    # randomly pick which 1,000 of the 2,000 users are in the "new" group; the first 110 users are the converters
    new_group = random.sample(range(2 * users_per_group), users_per_group)
    new_conversions = sum(1 for user in new_group if user < total_conversions)
    if abs(new_conversions - (total_conversions - new_conversions)) >= observed_gap:
        as_extreme += 1

print(f"In {runs:,} random shuffles, a gap this large happened {as_extreme / runs:.1%} of the time")
print("-> luck alone produces a gap this big about a third of the time, so this is the p-value idea in action.")
print("   (The z-test in T2 gives p = 0.33 and this shuffle test about 0.37; the methods differ slightly, but both say: not significant.)")

In 5,000 random shuffles, a gap this large happened 37.2% of the time
-> luck alone produces a gap this big about a third of the time, so this is the p-value idea in action.
   (The z-test in T2 gives p = 0.33 and this shuffle test about 0.37; the methods differ slightly, but both say: not significant.)


## T2. Read the A/B test

Two-proportion z-test, 95% confidence interval for the difference, and the sample size needed to detect a 1-point lift with 80% power.

In [12]:
def norm_cdf(z):
    """Standard normal cumulative distribution function (no scipy needed)."""
    return 0.5 * (1 + math.erf(z / math.sqrt(2)))

n_a, x_a = 1000, 50
n_b, x_b = 1000, 60
p_a, p_b = x_a / n_a, x_b / n_b

pooled = (x_a + x_b) / (n_a + n_b)
se_pooled = math.sqrt(pooled * (1 - pooled) * (1 / n_a + 1 / n_b))
z = (p_b - p_a) / se_pooled
p_value = 2 * (1 - norm_cdf(abs(z)))

se_diff = math.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
ci_low, ci_high = (p_b - p_a) - 1.96 * se_diff, (p_b - p_a) + 1.96 * se_diff

z_alpha, z_power = 1.96, 0.8416          # 5% two-sided, 80% power
n_needed = (z_alpha + z_power) ** 2 * (p_a * (1 - p_a) + p_b * (1 - p_b)) / (p_b - p_a) ** 2

print(f"Conversion: current {p_a:.1%}, new {p_b:.1%} (relative lift {(p_b / p_a - 1):+.0%}, absolute {p_b - p_a:+.1%})")
print(f"z = {z:.2f}, p-value = {p_value:.2f}")
print(f"95% CI for the difference: {ci_low:+.1%} to {ci_high:+.1%}")
print(f"Users needed per variant to detect a 1-point lift: about {n_needed:,.0f}")
assert round(z, 2) == 0.98 and round(p_value, 2) == 0.33 and round(n_needed, -2) == 8200

Conversion: current 5.0%, new 6.0% (relative lift +20%, absolute +1.0%)
z = 0.98, p-value = 0.33
95% CI for the difference: -1.0% to +3.0%
Users needed per variant to detect a 1-point lift: about 8,155


## Part 5. Falling conversion

Conversion by channel, the mix of traffic, and what October would have looked like with September's traffic mix.

In [13]:
traffic = pd.DataFrame(
    {"sep_sessions": [40000, 10000], "sep_orders": [2400, 200],
     "oct_sessions": [38000, 30000], "oct_orders": [2318, 630]},
    index=["Organic", "Paid Social"],
)
traffic["sep_conv_%"] = traffic["sep_orders"] / traffic["sep_sessions"] * 100
traffic["oct_conv_%"] = traffic["oct_orders"] / traffic["oct_sessions"] * 100
traffic["sep_share_%"] = traffic["sep_sessions"] / traffic["sep_sessions"].sum() * 100
traffic["oct_share_%"] = traffic["oct_sessions"] / traffic["oct_sessions"].sum() * 100
print(traffic.round(1).to_string())

sep_blended = traffic["sep_orders"].sum() / traffic["sep_sessions"].sum() * 100
oct_blended = traffic["oct_orders"].sum() / traffic["oct_sessions"].sum() * 100
oct_at_sep_mix = (traffic["oct_conv_%"] * traffic["sep_share_%"] / 100).sum()

print(f"\nBlended conversion: Sep {sep_blended:.2f}%, Oct {oct_blended:.2f}%")
print(f"October conversion if the traffic mix had stayed as in September: {oct_at_sep_mix:.2f}%")
print(f"Total orders: {traffic['sep_orders'].sum():,} -> {traffic['oct_orders'].sum():,} "
      f"({traffic['oct_orders'].sum() / traffic['sep_orders'].sum() - 1:+.1%})")
print(f"Organic sessions {38000 / 40000 - 1:+.1%}, organic orders {2318 / 2400 - 1:+.1%}; paid social orders {630 / 200 - 1:+.0%}")
assert round(sep_blended, 1) == 5.2 and round(oct_blended, 1) == 4.3 and round(oct_at_sep_mix, 1) == 5.3

             sep_sessions  sep_orders  oct_sessions  oct_orders  sep_conv_%  oct_conv_%  sep_share_%  oct_share_%
Organic             40000        2400         38000        2318         6.0         6.1         80.0         55.9
Paid Social         10000         200         30000         630         2.0         2.1         20.0         44.1

Blended conversion: Sep 5.20%, Oct 4.34%
October conversion if the traffic mix had stayed as in September: 5.30%
Total orders: 2,600 -> 2,948 (+13.4%)
Organic sessions -5.0%, organic orders -3.4%; paid social orders +215%


**Reading it:** both channels held steady or improved, so the blended fall comes from the mix shift (paid social went from 20% to 44% of sessions), not from the site getting worse. The recommendation to the Head of Growth is in `README.md`.